# 模块七（补）: baseline_v2 复评 — 公平对比前置步骤

## 为什么要复评？

v2 和 v3 直接对比存在**两类系统性偏差**，导致指标不可比：

1. **裁判不一致**：v2 用旧裁判（话题内容判定法），v3 用 v4.4 新裁判（行为特征判定法）。同一回复，两版裁判可能给出不同分数。
2. **用例错误**：v2 时期的部分用例期望设置错误（如 TC-INTENT-008 物流查询被错标为商品咨询），v3 已修正。

**解决办法**：保留一个固定量（v2 系统的 `actual_response`），只替换两个变量——用 **v4.4 裁判** + **v3 修正后的用例定义** 对 v2 的回复重新打分。

产出 `baseline_v2_v44.json`，与 `baseline_v3.json` 在**同裁判、同用例**下对比，对比的就是纯粹的系统行为差异。

## A. 环境配置与数据加载

In [17]:
# ========== 配置 ==========
import json, os, re, time, sys
from getpass import getpass
from typing import Optional, List, Dict
from openai import OpenAI
from collections import Counter

print("=== baseline_v2 复评 v4.4 ===")

# ---------- 输入/输出路径 ----------
DEFAULT_BASE = 'C:/Users/Darren/Desktop/ecommerce_rag_eval'

# v2 原始结果（复评源：取其 actual_response）
V2_PATH = input(f"v2 原始结果路径（默认 {DEFAULT_BASE}/eval_output_baseline_v2/eval_results_baseline_v2.json）: ").strip()
if not V2_PATH:
    V2_PATH = f"{DEFAULT_BASE}/eval_output_baseline_v2/eval_results_baseline_v2.json"

# v3 结果（用例定义源 + 多轮上下文源 + 对比参照）
V3_PATH = input(f"v3 评测结果路径（默认 {DEFAULT_BASE}/eval_output_baseline_v3/eval_results_baseline_v3_full_20260817_150803.json）: ").strip()
if not V3_PATH:
    V3_PATH = f"{DEFAULT_BASE}/eval_output_baseline_v3/eval_results_baseline_v3_full_20260817_150803.json"

# 复评结果输出目录
V2_OUTPUT_DIR = input(f"v2 结果输出目录（默认 {DEFAULT_BASE}/eval_output_baseline_v2）: ").strip()
if not V2_OUTPUT_DIR:
    V2_OUTPUT_DIR = f"{DEFAULT_BASE}/eval_output_baseline_v2"
os.makedirs(V2_OUTPUT_DIR, exist_ok=True)

# ---------- 裁判模型配置（务必与 v3 评测一致，保证裁判等价）----------
JUDGE_API_KEY = getpass("裁判 LLM API Key（与 v3 评测一致）: ")

# 阿里云百炼（v3 评测使用的裁判，默认采用同型号）
ALIYUN_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
use_aliyun = input(f"是否使用阿里云百炼（{ALIYUN_BASE_URL}）？回车确认，输入 n 自定义: ").strip().lower()
if use_aliyun in ('n', 'no', '否'):
    JUDGE_BASE_URL = input("自定义 Base URL: ").strip()
else:
    JUDGE_BASE_URL = ALIYUN_BASE_URL

JUDGE_MODEL = input("裁判模型（默认 qwen-plus，与 v3 评测一致）: ").strip()
if not JUDGE_MODEL:
    JUDGE_MODEL = "qwen-plus"

judge_client = OpenAI(base_url=JUDGE_BASE_URL, api_key=JUDGE_API_KEY)

print(f"\n配置完成")
print(f"   v2 结果: {V2_PATH}")
print(f"   v3 结果: {V3_PATH}")
print(f"   输出目录: {V2_OUTPUT_DIR}")
print(f"   裁判 BaseURL: {JUDGE_BASE_URL}")
print(f"   裁判模型: {JUDGE_MODEL}")

# ---------- 连通性自检（防止全 0 分坑）----------
print("\n>>> 裁判连通性自检...")
_ok = False
try:
    _resp = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": "请回复：OK"}],
        temperature=0
    )
    _content = _resp.choices[0].message.content
    print(">>> 连通正常，裁判返回:", _content)
    _ok = True
except Exception as _e:
    print(">>> 裁判连接失败:", _e)
    print(">>> 请检查 JUDGE_BASE_URL / JUDGE_API_KEY / JUDGE_MODEL 后再继续，否则会产生全 0 分结果！")

if not _ok:
    raise SystemExit("裁判连接失败，已终止。请修正配置后重新运行本单元格。")

=== baseline_v2 复评 v4.4 ===


v2 原始结果路径（默认 C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v2/eval_results_baseline_v2.json）:  
v3 评测结果路径（默认 C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v3/eval_results_baseline_v3_full_20260817_150803.json）:  
v2 结果输出目录（默认 C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v2）:  
裁判 LLM API Key（与 v3 评测一致）:  ········
是否使用阿里云百炼（https://dashscope.aliyuncs.com/compatible-mode/v1）？回车确认，输入 n 自定义:  
裁判模型（默认 qwen-plus，与 v3 评测一致）:  



配置完成
   v2 结果: C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v2/eval_results_baseline_v2.json
   v3 结果: C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v3/eval_results_baseline_v3_full_20260817_150803.json
   输出目录: C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v2
   裁判 BaseURL: https://dashscope.aliyuncs.com/compatible-mode/v1
   裁判模型: qwen-plus

>>> 裁判连通性自检...
>>> 连通正常，裁判返回: OK


### A.1 加载 V2 与 V3 数据

In [18]:
# ========== 加载 v2 原始结果（复评源：取其 actual_response）==========
with open(V2_PATH, 'r', encoding='utf-8') as f:
    v2_raw = json.load(f)

print(f'v2 原始结果: {len(v2_raw)} 条')
print(f'  缺 actual_response: {sum(1 for r in v2_raw if not r.get("actual_response"))} 条')

# ========== 加载 v3 结果（用例定义源 + 多轮上下文源 + 对比参照）==========
with open(V3_PATH, 'r', encoding='utf-8') as f:
    v3_raw = json.load(f)

# v3 按 case_id 建立索引
v3_by_id = {r['case_id']: r for r in v3_raw}
print(f'v3 结果: {len(v3_raw)} 条')

# ========== 校验 case_id 一一对应 ==========
v2_ids = set(r['case_id'] for r in v2_raw)
v3_ids = set(v3_by_id.keys())
print(f'仅 v2 有: {sorted(v2_ids - v3_ids)}')
print(f'仅 v3 有: {sorted(v3_ids - v2_ids)}')
print(f'交集条数: {len(v2_ids & v3_ids)} (应等于 505)')

v2 原始结果: 505 条
  缺 actual_response: 0 条
v3 结果: 505 条
仅 v2 有: []
仅 v3 有: []
交集条数: 505 (应等于 505)


## B. v4.4 裁判函数

### B.1 确定性预检辅助函数

`check_must_not`(精确子串)、`_is_range_match`(范围语义)、`check_expected_points`(宽松匹配) 与 v4.4 版完全一致。

In [19]:
# ========== 确定性预检: must_not 精确子串匹配 ==========
def check_must_not(response, must_not_list):
    if not must_not_list:
        return False
    for item in must_not_list:
        item = item.strip()
        if not item:
            continue
        if item in response:
            return True
    return False

# ========== 范围语义匹配辅助函数 ==========
def _is_range_match(point, response):
    m = re.match(r'^(\d+)\s*[-~～]\s*(\d+)$', point.strip())
    if not m:
        return False
    p_start = int(m.group(1)); p_end = int(m.group(2))
    resp_ranges = re.findall(r'(\d+)\s*[-~～]\s*(\d+)\s*(个工作日|天|小时|分钟)?', response)
    if not resp_ranges:
        return False
    all_starts = [int(rr[0]) for rr in resp_ranges]
    all_ends = [int(rr[1]) for rr in resp_ranges]
    if all_starts and all_ends:
        if min(all_starts) <= p_start and max(all_ends) >= p_end:
            return True
    for rs, rre, unit in resp_ranges:
        if int(rs) <= p_start and int(rre) >= p_end:
            return True
    return False

# ========== 确定性预检: expected_points 宽松匹配 ==========
def check_expected_points(response, expected_points):
    if not expected_points:
        return True
    for point in expected_points:
        point = point.strip()
        if not point:
            continue
        if point in response:
            continue
        point_clean = point.replace("，", "").replace("。", "").replace("、", "").replace(" ", "")
        resp_clean = response.replace(" ", "")
        if point_clean in resp_clean:
            continue
        if _is_range_match(point, response):
            continue
        if len(point) >= 4:
            if point[:4] in response:
                continue
        num_match = re.match(r'^(\d+)(.+)$', point)
        if num_match:
            num = num_match.group(1); unit = num_match.group(2)
            pattern = num + r'[^0-9]{0,5}' + re.escape(unit[-1])
            if re.search(pattern, response):
                continue
        return False
    return True

### B.2 v4.4 裁判 llm_judge

In [20]:
# ========== 裁判评分（v4.4 复评版）==========
def llm_judge(case: dict, actual_response: str) -> dict:
    must_not_str = "、".join(case.get('must_not', [])) if case.get('must_not') else "无"
    expected_points = "、".join(case.get('expected_answer_contains', [])) if case.get('expected_answer_contains') else "无特殊要求"
    context_str = ""
    if case.get('conversation_context'):
        lines = []
        for t in case['conversation_context']:
            role = "用户" if t.get('role') == 'user' else "客服"
            lines.append(f"{role}: {t.get('content', '')[:80]}")
        context_str = "\n".join(lines)
    complaint_level = case.get('expected_complaint_level') or "不适用"
    path_str = case.get('expected_if_else_path')
    if isinstance(path_str, list):
        path_str = ', '.join(path_str)

    prompt = f"""你是一名严格的电商客服质量评审专家。请评估以下客服回复的质量。

## 评分规则（必须严格遵守）

### branch_routing（分支路由）评分规则
- 你无法看到系统的实际路由 trace，请根据回复的【行为模式】判断，而非回复的【话题内容】
- 判定方法：观察回复在"做什么"，而不是回复"在讲什么"
- 5分：回复的行为模式与期望分支"{case['expected_branch']}"完全一致
- 3-4分：行为模式基本符合期望分支，但有少量偏差
- 0-2分：行为模式明显属于其他分支

【各分支行为特征对照表】（核心判定依据）

1. 商品咨询与规则查询分支（知识检索 + LLM直接回答）：
   - 行为特征：直接解答用户的政策/规则/参数问题，不引导任何操作流程
   - 典型表现：直接给出答案（时效/运费/规则/参数等）、引用知识库内容、不索要订单号、不确认办理意向
   - 注意：即使回答的是"退款多久到账""退换货政策"等售后话题，只要是直接解答规则而非引导办理，就属于此分支
   - 操作流程咨询也属此分支：用户问"怎么操作/怎么申请/流程是什么"，回复描述操作步骤/方法 → 商品咨询（解答方法）
   - 知识库未命中也属此分支：回复"暂未收录/知识库没有相关信息" → 路由仍是商品咨询，只是信息完整性低

2. 退款换货分支（售后办理流程）：
   - 行为特征：引导用户完成退款/退货/换货/取消/修改的办理流程
   - 典型表现：确认退款/退货/换货意向、索要订单号、询问退款原因、引导寄回、告知办理步骤
   - 注意：如果只是回答退款政策问题（如时效），不引导办理，则不属于此分支
   - 告知办理结果也属此分支：如查询订单后告知"已超期无法办理""已签收无法取消" → 退款换货分支（已处理并给出结果）
   - 子场景包括：退款、退货、换货、取消订单、修改订单、价格保护申请等所有售后办理动作

3. 订单查询分支（工具调用查订单）：
   - 行为特征：查询并展示具体订单的实时数据
   - 典型表现：提供订单状态、物流信息、具体订单数据，通常需要订单号
   - 索要订单号以进行查询也属此分支
   - 注意：即使回复内容有幻觉（虚构物流信息），只要行为模式是在告知订单/物流状态，分支路由仍应判定为订单查询（幻觉由 faithfulness 维度负责）

4. 投诉分支（投诉处理流程）：
   - 行为特征：处理投诉/追责/赔偿诉求
   - 典型表现：道歉、记录投诉、升级处理、赔偿方案、安抚情绪
   - 投诉分支也可以转人工：作为升级处理的一种方式，不代表路由错误
   - 判断标准：是否有投诉处理的核心行为（道歉+记录/升级/补偿），转人工只是附加动作

5. 其他分支（兜底/转人工）：
   - 行为特征：无法归类、转人工、或通用兜底回复
   - 典型表现："抱歉我无法回答"、"为您转接人工"、身份说明
   - 注意：如果回复有明确的业务分支行为（如道歉+补偿=投诉、解答规则=商品咨询），即使附带转人工，也不应判为其他分支

【关键判定原则】
- 话题内容 ≠ 分支路由。退款话题可能走商品咨询分支（咨询政策），也可能走退款换货分支（办理退款）
- 核心区分：回复是在"提供信息/解答规则"还是"引导操作/办理流程"
- 直接解答政策/规则问题 → 商品咨询分支；引导办理/操作流程 → 对应业务分支
- 只要行为模式符合期望分支，即使回复内容涉及其他分支话题，也应给4分以上

【核心行为优先原则】（重要，避免误判）
- 当回复包含多种行为时，以【核心/主要行为】判定分支，不以附带/次要行为判定
- 判断方法：看回复的主体内容在做什么，开头和结尾的附加引导不算核心
- 示例 1：用户问"换货多久能收到"，回复先解答"3-5个工作日"（核心：规则解答），最后说"请提供订单号进一步核实"（附加：引导办理）→ 核心是商品咨询，branch_routing 4-5分
- 示例 2：用户问"质量问题能免费修吗"，回复先解答"质保期内非人为可免费修"（核心：规则解答），后半段引导"申请售后上传照片"（附加：引导办理）→ 核心是商品咨询，branch_routing 4-5分
- 示例 3：用户问"退货运费谁承担"，回复先解答"质量问题商家承担，无理由用户承担"（核心：规则解答），最后"请确认退货原因"（附加：轻微引导）→ 核心是商品咨询，branch_routing 4-5分

【操作流程咨询特殊判定】（重要，避免误判）
- 用户问"怎么操作/怎么申请/流程是什么/如何办理"等带疑问词的问题 → 属于操作流程咨询
- 回复描述了操作步骤/方法（如"点击申请售后→选择价格保护"）→ 商品咨询分支（解答方法）
- 回复引导用户现在就去办理（如"请提供订单号我帮您申请"）→ 退款换货分支（引导办理）
- 区分标准：回复是在"告诉用户怎么操作"（咨询）还是在"引导用户现在就办"（办理）
- 示例 1：用户问"怎么申请价格保护"，回复"在订单详情页点击申请售后→选择价格保护" → 商品咨询（解答方法）
- 示例 2：用户说"我要申请价格保护"，回复"请提供订单号" → 退款换货（引导办理）
- 示例 3：用户问"七天无理由退货怎么操作"，回复列出操作步骤 → 商品咨询（解答方法）

【大类与子场景区分原则】（重要，避免误判）
- 退款换货是一个大类，包含多个子场景：退款、退货、换货、修改订单、取消订单等
- branch_routing 只评判【大类】是否正确，不要求子场景完全匹配
- 只要回复的行为模式属于"售后办理引导"（确认需求、索要信息、引导流程、告知办理条件、告知办理结果等），即视为退款换货大类路由正确
- 子场景是否精准匹配（如改地址 vs 换货、取消订单 vs 退款），由 intent_accuracy 维度负责评分，branch_routing 不因此扣分
- 示例 1：用户说"改地址"，系统回复"已签收无法改地址，可考虑退货重拍" → branch_routing 4-5分（大类正确），intent_accuracy 可酌情扣分
- 示例 2：用户说"修改订单"，系统回复"请问要修改什么内容" → branch_routing 4-5分（大类正确），intent_accuracy 可酌情扣分
- 示例 3：用户说"我要退款"，系统回复"请问退款原因" → branch_routing 5分（大类和子场景都正确）

【超期/不符合条件告知的判定】（避免误判）
- 用户申请售后（退款/退货/换货），系统查询后告知"已超期/不符合条件/无法办理" → 属于退款换货分支的正确行为
- 系统已经处理了用户的诉求（查询订单+给出结论），只是结论是"无法办理"
- 不能因为系统没有"引导办理流程"就扣分，因为不符合条件时本就不应引导办理
- 示例：用户说"退订单 3178642857123451"，回复"已签收超过7天，无法取消，质量问题可申请质保" → branch_routing 4-5分（大类正确，已处理并给出结果）

【路由维度与其他维度的分离原则】（极其重要）
- branch_routing 只评判【行为模式属于哪个分支】，不评判内容的真实性、完整性、准确性
- 内容真实性 → faithfulness 维度负责
- 内容完整性 → information_completeness 维度负责
- 意图准确性 → intent_accuracy 维度负责
- 示例 1：订单查询分支回复了虚构的物流信息 → branch_routing 应高分（路由正确），faithfulness 应低分（幻觉）
- 示例 2：商品咨询分支回复"暂未收录" → branch_routing 应高分（路由正确），information_completeness 可低分（信息不足）
- 示例 3：投诉分支最后转了人工 → branch_routing 应高分（投诉行为是核心），不能因为转人工就判其他分支

### must_not_violation（禁止表述违规）判定规则
- 逐条检查 must_not 列表，使用精确子串匹配
- 只有当回复中完整包含 must_not 中的某个表述时，才判定为违规
- 禁止使用语义近似、同义词扩展来判断
- 否定表述不算违规：如 must_not=["可办理退货"]，回复中有"不可办理退货"→不违规（语义相反）
- 不同表述不算违规：如 must_not=["7天内可退"]，回复中有"7天无理由退货"→不违规（不同表述）

### contains_expected_points（要点覆盖）判定规则
- 逐条检查 expected_answer_contains 中的每个要点
- 使用宽松匹配：关键词的变形、包含该关键词的 longer phrase 均算命中
  如"未签收"匹配"尚未签收"、"运输中"匹配"目前运输中"
- 范围匹配：如期望"1-7天"，回复中有"1-3天"和"3-7天"等分段范围整体覆盖1-7区间，也视为命中
- 只要回复中体现了该要点的核心语义，即算命中
- 全部命中→true，有任何一个未命中→false

### boundary_correctness（边界正确性）评分规则
- 根据回复是否正确处理了边界值说明中的业务逻辑来评分
- 只要回复体现了边界值对应的核心处理逻辑，即给4分以上
- 不要因为回复提供了额外的替代方案就扣分

## 用例信息
【用例ID】{case['case_id']}
【测试类型】{case.get('test_type', '')}
【用户问题（最后一轮）】{case.get('user_query', '')}
【对话上下文】
{context_str if context_str else '无（单轮对话）'}

【期望意图】{case.get('expected_intent', '')}
【期望分支】{case.get('expected_branch', '')}
【期望IF/ELSE路径】{path_str or '无'}
【期望答案应包含的关键要点】{expected_points}
【回复中禁止出现的表述】{must_not_str}
【期望工具调用】{', '.join(case.get('expected_tool_calls', [])) if case.get('expected_tool_calls') else '无'}
【期望投诉分级】{complaint_level}
【是否期望转人工】{case.get('expected_human_handoff', False)}
【边界值说明】{case.get('boundary_value') or '无'}

【实际客服回复】
{actual_response}

请从以下维度严格打分（0-5分），并输出 JSON 格式：
{{
    "intent_accuracy": <0-5>,
    "branch_routing": <0-5>,
    "answer_relevance": <0-5>,
    "information_completeness": <0-5>,
    "faithfulness": <0-5>,
    "tone_compliance": <0-5>,
    "must_not_violation": <true/false>,
    "boundary_correctness": <0-5>,
    "is_hallucination": <true/false>,
    "contains_expected_points": <true/false>,
    "handoff_correctness": <0-5, 不适用填-1>,
    "comment": "<80字以内>"
}}

只输出 JSON。"""

    try:
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e1:
        try:
            resp = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            content = resp.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("\n", 1)[1] if "\n" in content else content[3:]
            if content.endswith("```"):
                content = content.rsplit("```", 1)[0]
            content = content.strip()
            if content.startswith("json"):
                content = content[4:].strip()
            return json.loads(content)
        except Exception:
            return {
                "error": str(e1), "intent_accuracy": 0, "branch_routing": 0,
                "answer_relevance": 0, "information_completeness": 0,
                "faithfulness": 0, "tone_compliance": 0,
                "must_not_violation": False, "boundary_correctness": 0,
                "is_hallucination": False, "contains_expected_points": False,
                "handoff_correctness": -1, "comment": f"裁判调用失败: {e1}"
            }

print('v4.4 裁判函数定义完成')

v4.4 裁判函数定义完成


## C. 用例对齐：用 v3 的修正后用例定义 + 上下文

In [21]:
# ========== 用例对齐 ==========
# 规则：对每个 v2 case，取其 actual_response；用例定义(expected_*)、多轮上下文从 v3 取
# 这样保证 v2 与 v3 在相同期望、相同上下文的条件下比较

def normalize_points(v):
    # 把 str('降噪,42dB') 或 list 统一成 list
    if isinstance(v, list):
        return [x.strip() for x in v if str(x).strip()]
    if isinstance(v, str):
        v = v.strip()
        if not v:
            return []
        return [x.strip() for x in v.replace('，', ',').split(',') if x.strip()]
    return []

def build_rerun_case(v2_rec, v3_ref):
    # v2 的回复 + v3 的用例定义 + v3 的多轮上下文
    case = dict(v3_ref)  # 以 v3 的定义为基底（含修正后的期望、context）
    case['actual_response'] = v2_rec.get('actual_response', '')
    case['_source'] = 'v2_rerun'
    if 'score' in case:
        case.pop('score')  # 去掉旧 score，换成复评
    return case

# 构建统一 case 列表
rerun_cases = []
skipped = []
for v2_rec in v2_raw:
    cid = v2_rec['case_id']
    v3_ref = v3_by_id.get(cid)
    if v3_ref is None:
        skipped.append(cid)
        continue
    rerun_cases.append(build_rerun_case(v2_rec, v3_ref))

print(f'可复评用例: {len(rerun_cases)} 条')
print(f'跳过（v3 无对应）: {skipped}')

# 校验对齐后的分支分布
from collections import Counter
print()
print('复评用例分支分布:')
for k, v in Counter(c.get('expected_branch', '?') for c in rerun_cases).items():
    print(f'  {k}: {v}')

# 检查多轮用例（从 v3 借的上下文）
multi = [c['case_id'] for c in rerun_cases if c.get('conversation_context')]
print()
print(f'多轮用例（含 context）: {len(multi)} 条: {multi}')

可复评用例: 505 条
跳过（v3 无对应）: []

复评用例分支分布:
  商品咨询与规则查询: 280
  订单查询: 42
  退款换货: 82
  投诉: 52
  其他: 49

多轮用例（含 context）: 0 条: []


## D. 断点续评

In [22]:
# ========== 断点续评 ==========
CHECKPOINT = os.path.join(V2_OUTPUT_DIR, 'rerun_checkpoint.json')

def load_checkpoint():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(res):
    with open(CHECKPOINT, 'w', encoding='utf-8') as f:
        json.dump(res, f, ensure_ascii=False, indent=2)

print('断点续评就绪，检查点:')
print(' ', CHECKPOINT)

断点续评就绪，检查点:
  C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v2\rerun_checkpoint.json


## E. 批量复评（505 次裁判调用，约需数分钟）

In [23]:
# ========== 批量复评 ==========
# 每 20 条自动保存检查点，中途中断可重新运行本单元格续评
BATCH_SAVE = 20
SLEEP = 0.0

results = load_checkpoint()  # {case_id: {record}}
done_ids = set(results.keys())

# 先按断点续评整合
rerun_results = []

print(f'开始复评: 总计 {len(rerun_cases)} 条，已完成 {len(done_ids)} 条，待评 {len(rerun_cases)-len(done_ids)} 条')

# 1. 先载入已完成结果
for cid, rec in results.items():
    rerun_results.append(rec)

try:
    for idx, case in enumerate(rerun_cases):
        cid = case['case_id']
        if cid in done_ids:
            continue
        score = llm_judge(case, case['actual_response'])

        # 确定性预检覆盖裁判误判
        must_list = case.get('must_not', [])
        pt_list = case.get('expected_answer_contains', [])
        if not check_must_not(case['actual_response'], must_list) and score.get('must_not_violation', False):
            score['must_not_violation'] = False
        if check_expected_points(case['actual_response'], pt_list) and not score.get('contains_expected_points', False):
            score['contains_expected_points'] = True

        rec = {
            'case_id': cid,
            'test_type': case.get('test_type', ''),
            'category': case.get('category', ''),
            'difficulty': case.get('difficulty', ''),
            'user_query': case.get('user_query', ''),
            'expected_intent': case.get('expected_intent', ''),
            'expected_branch': case.get('expected_branch', ''),
            'expected_answer_contains': case.get('expected_answer_contains', []),
            'must_not': case.get('must_not', []),
            'expected_complaint_level': case.get('expected_complaint_level'),
            'expected_human_handoff': case.get('expected_human_handoff', False),
            'is_multi_turn': bool(case.get('conversation_context')),
            'actual_response': case['actual_response'],
            'score': score,
        }
        rerun_results.append(rec)
        results[cid] = rec

        if (len(results)) % BATCH_SAVE == 0:
            save_checkpoint(results)
            print(f'  已复评 {len(results)}/{len(rerun_cases)}，检查点已保存')

        if SLEEP:
            time.sleep(SLEEP)

except KeyboardInterrupt:
    save_checkpoint(results)
    print(f'手动中断，已保存 {len(results)} 条检查点')

# 完成后保存最终结果
save_checkpoint(results)
print(f'复评完成: 共 {len(rerun_results)} 条')

# 输出到统一文件
OUT = os.path.join(V2_OUTPUT_DIR, 'baseline_v2_v44.json')
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(rerun_results, f, ensure_ascii=False, indent=2)
print(f'已保存: {OUT}')

开始复评: 总计 505 条，已完成 0 条，待评 505 条
  已复评 20/505，检查点已保存
  已复评 40/505，检查点已保存
  已复评 60/505，检查点已保存
  已复评 80/505，检查点已保存
  已复评 100/505，检查点已保存
  已复评 120/505，检查点已保存
  已复评 140/505，检查点已保存
  已复评 160/505，检查点已保存
  已复评 180/505，检查点已保存
  已复评 200/505，检查点已保存
  已复评 220/505，检查点已保存
  已复评 240/505，检查点已保存
  已复评 260/505，检查点已保存
  已复评 280/505，检查点已保存
  已复评 300/505，检查点已保存
  已复评 320/505，检查点已保存
  已复评 340/505，检查点已保存
  已复评 360/505，检查点已保存
  已复评 380/505，检查点已保存
  已复评 400/505，检查点已保存
  已复评 420/505，检查点已保存
  已复评 440/505，检查点已保存
  已复评 460/505，检查点已保存
  已复评 480/505，检查点已保存
  已复评 500/505，检查点已保存
复评完成: 共 505 条
已保存: C:/Users/Darren/Desktop/ecommerce_rag_eval/eval_output_baseline_v2\baseline_v2_v44.json


## F. 复评结果校验 -- 确认公平对比可用

In [25]:
# ========== 复评结果校验 ==========
OUT = os.path.join(V2_OUTPUT_DIR, 'baseline_v2_v44.json')
with open(OUT, 'r', encoding='utf-8') as f:
    v2_v44 = json.load(f)

print(f're-run 结果: {len(v2_v44)} 条')

# ---- 安全校验：只要有 error 字段即视为失败，必须告警 ----
_err_cases = [r['case_id'] for r in v2_v44 if 'error' in r.get('score', {})]
if _err_cases:
    print(f"!!! 检测到 {len(_err_cases)} 条判分为裁判失败（含 error 字段）!!!")
    print(f"!!! 例: {_err_cases[:10]}")
    print(f"!!! 这是全 0 分的错误结果，请删除检查点和结果文件后重跑，不可用于对比！")
    raise SystemExit("复评存在裁判失败，已终止校验。请清缓存重跑。")

print("安全校验通过：无裁判失败记录 ✅")

score_cols = ['intent_accuracy', 'branch_routing', 'answer_relevance',
              'information_completeness', 'faithfulness', 'tone_compliance', 'boundary_correctness']

pass_count = sum(1 for r in v2_v44 if r['score'].get('branch_routing', 0) >= 4)

print()
print('=== V2 复评（v4.4裁判 + v3用例）汇总 ===')
print(f'  通过率: {pass_count}/{len(v2_v44)} = {pass_count/len(v2_v44)*100:.1f}%')
for col in score_cols:
    vals = [r['score'].get(col, 0) for r in v2_v44]
    print(f'  {col}: {sum(vals)/len(vals):.2f}')

# 对比 v3（同裁判、同用例）
v3_pass = sum(1 for r in v3_raw if r.get('score', {}).get('branch_routing', 0) >= 4)
print()
print('=== 公平对比: V2复评 vs V3 ===')
print(f'  V2(复评) 通过率: {pass_count/len(v2_v44)*100:.1f}%')
print(f'  V3        通过率: {v3_pass/len(v3_raw)*100:.1f}%')
print(f'  真实提升: {v3_pass/len(v3_raw)*100 - pass_count/len(v2_v44)*100:+.1f}pp')
print()
print('⚠️ 注意：本节通过率与 v2 原始评测（旧裁判/旧用例）不直接可比，务必用复评后的 v2_v44 做对比！')

re-run 结果: 505 条
安全校验通过：无裁判失败记录 ✅

=== V2 复评（v4.4裁判 + v3用例）汇总 ===
  通过率: 476/505 = 94.3%
  intent_accuracy: 4.73
  branch_routing: 4.74
  answer_relevance: 4.76
  information_completeness: 4.00
  faithfulness: 4.61
  tone_compliance: 4.94
  boundary_correctness: 4.91

=== 公平对比: V2复评 vs V3 ===
  V2(复评) 通过率: 94.3%
  V3        通过率: 97.0%
  真实提升: +2.8pp

⚠️ 注意：本节通过率与 v2 原始评测（旧裁判/旧用例）不直接可比，务必用复评后的 v2_v44 做对比！
